# Spendende für *CharityML* finden

In diesem Projekt testen wir mehrere Algorithmen für überwachtes Lernen, um das Einkommen von Personen anhand von Daten aus dem US-Zensus von 1994 treffsicher zu modellieren. Danach wählen wir anhand vorläufiger Ergebnisse den besten Kandidaten unter den Algorithmen aus und optimieren diesen Algorithmus weiter, um die Daten bestmöglich zu modellieren. Unser Ziel bei dieser Implementierung ist es, ein Modell zu konstruieren, das treffsicher vorhersagt, ob eine Person mehr als 50.000 $ verdient. Eine solche Aufgabe kann im gemeinnützigen Bereich auftreten, wo Organisationen von Spenden leben.  Das Einkommen einer Person zu kennen, kann einer gemeinnützigen Organisation helfen, besser einzuschätzen, wie hoch die Spende sein sollte, um die sie bittet, oder ob sie überhaupt Kontakt aufnehmen sollte.  Auch wenn es schwierig sein kann, die allgemeine Einkommensklasse einer Person direkt aus öffentlichen Quellen zu bestimmen, können wir diesen Wert aus anderen öffentlich verfügbaren Features ableiten.

Der Datensatz für dieses Projekt stammt aus dem [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Census+Income). Ron Kohavi und Barry Becker haben den Datensatz zur Verfügung gestellt, nachdem er im Artikel _„Scaling Up the Accuracy of Naive-Bayes Classifiers: A Decision-Tree Hybrid“_ veröffentlicht worden war. Du findest den Artikel von Ron Kohavi [online](https://www.aaai.org/Papers/KDD/1996/KDD96-033.pdf). Die Daten, die wir hier untersuchen, enthalten kleine Änderungen gegenüber dem ursprünglichen Datensatz, etwa das Entfernen des Features `'fnlwgt'` und von Einträgen mit fehlenden oder falsch formatierten Werten.

## Lernziele

Am Ende dieses Notebooks kannst du:

- gemischte Zensusdaten mit Log-Transformationen, One-Hot Encoding und Feature-Skalierung vorverarbeiten.
- eine naive Baseline festlegen und eine passende Evaluationsmetrik wählen (den F-beta-Score).
- einen Entscheidungsbaum und einen AdaBoost-Klassifikator anhand von Accuracy und F-Score vergleichen.
- das stärkste Modell mit Grid Search tunen und die Auswirkung eines verkleinerten Feature-Sets beurteilen.

----
## Daten erkunden

Beginnen wir damit, die nötigen Bibliotheken zu importieren, die Daten einzulesen und uns den Datensatz anzusehen.

Beachte, dass die letzte Spalte dieses Datensatzes, `'income'`, unsere Zielvariable sein wird (ob eine Person jährlich mehr als oder höchstens 50.000 $ verdient). Alle anderen Spalten sind Features zu jeder Person in der Zensus-Datenbank.

In [ ]:
# Für dieses Projekt benötigte Bibliotheken importieren

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Funktionen aus unserer Python-Datei importieren (siehe visuals_script.py im Repo)
import visuals_script as vs

# Schönere Anzeige in Notebooks
%matplotlib inline

sns.set(style="whitegrid", color_codes=True)

Beginnen wir damit, den Datensatz zu importieren und die ersten Zeilen zu untersuchen. 

In [ ]:
# Zensus-Datensatz laden
data = 

In [ ]:
# Datentypen untersuchen

Eine einfache Untersuchung des Datensatzes kann zeigen, wie viele Personen in die eine oder die andere Gruppe fallen, und uns den Prozentsatz der Personen nennen, die mehr als 50.000 \$ verdienen.

Sehen wir uns Folgendes an:

- Die Gesamtzahl der Einträge, `'n_records'`
- Die Anzahl der Personen, die jährlich mehr als 50.000 \$ verdienen, `'n_greater_50k'`.
- Die Anzahl der Personen, die jährlich höchstens 50.000 \$ verdienen, `'n_at_most_50k'`.
- Der Prozentsatz der Personen, die jährlich mehr als 50.000 \$ verdienen, `'greater_percent'`.

In [ ]:
# TODO: Gesamtzahl der Einträge
n_records = 

# TODO: Anzahl der Einträge, bei denen das Einkommen der Person über 50.000 $ liegt
n_greater_50k = 

# TODO: Anzahl der Einträge, bei denen das Einkommen der Person höchstens 50.000 $ beträgt
n_at_most_50k = 

# TODO: Anteil der Personen mit einem Einkommen über 50.000 $ in Prozent
greater_percent = 

# Ergebnisse ausgeben
print ("Total number of records: {}".format(n_records))
print ("Individuals making more than $50,000: {}".format(n_greater_50k))
print ("Individuals making at most $50,000: {}".format(n_at_most_50k))
print ("Percentage of individuals making more than $50,000: {:.2f}%".format(greater_percent))

Wir können auch den Zusammenhang zwischen verschiedenen Features einer Person und ihrem Einkommen visualisieren.

Sehen wir uns an, wie sich die Anzahl der Menschen, die über oder unter 50K verdienen, nach Geschlecht und Bildungsniveau aufteilt.

In [ ]:
# Eine Möglichkeit, den Zusammenhang zu visualisieren, ist ein catplot von seaborn

----
## Daten vorbereiten
Bevor Daten als Eingabe für Machine-Learning-Algorithmen verwendet werden können, müssen sie oft bereinigt, formatiert und umstrukturiert werden. Dieser Schritt heißt üblicherweise **Vorverarbeitung**. Glücklicherweise gibt es in diesem Datensatz keine ungültigen oder fehlenden Einträge, mit denen wir umgehen müssen, allerdings haben bestimmte Features einige Eigenschaften, die angepasst werden müssen. Diese Vorverarbeitung kann das Ergebnis und die Vorhersagekraft fast aller Lernalgorithmen enorm verbessern.

In [ ]:
# Bevor wir weitermachen, teilen wir die Daten in Features und Zielvariable auf
features = 
target = 

### Schiefe stetige Features transformieren
Ein Datensatz kann manchmal mindestens ein Feature enthalten, dessen Werte meist nahe bei einer einzigen Zahl liegen, das aber auch eine nicht unerhebliche Anzahl von Werten hat, die weit größer oder kleiner als diese Zahl sind.  Algorithmen können empfindlich gegenüber solchen Werteverteilungen sein und schlechter abschneiden, wenn der Wertebereich nicht richtig normalisiert ist. Im Zensus-Datensatz passen zwei Features zu dieser Beschreibung: '`capital-gain'` und `'capital-loss'`.

Stellen wir ein Histogramm dieser beiden Features dar und sehen wir uns an, wie sie verteilt sind. (Schau dir das Python-Skript in diesem Repo an, wenn dich der Code für diese Plot-Funktion interessiert.)

In [ ]:
# Schiefe stetige Features der Originaldaten visualisieren
vs.distribution(data)

Bei stark schiefen Feature-Verteilungen wie `'capital-gain'` und `'capital-loss'` ist es gängige Praxis, eine <a href="https://en.wikipedia.org/wiki/Data_transformation_(statistics)">logarithmische Transformation</a> auf die Daten anzuwenden, damit die sehr großen und sehr kleinen Werte die Leistung eines Lernalgorithmus nicht negativ beeinflussen. Eine logarithmische Transformation verkleinert den Wertebereich, den Ausreißer verursachen, deutlich.

In [ ]:
# Log-Transformation der schiefen Features ['capital-gain', 'capital-loss']

# Neue logarithmierte Verteilungen visualisieren
vs.distribution(features, transformed=True)

### Datenvorverarbeitung

In der Tabelle unter **Daten erkunden** oben sehen wir, dass es für jeden Eintrag mehrere nicht numerische Features gibt. Lernalgorithmen erwarten normalerweise numerische Eingaben, deshalb müssen nicht numerische Features (sogenannte *kategoriale Variablen*) umgewandelt werden. Eine beliebte Methode, kategoriale Variablen umzuwandeln, ist das **One-Hot Encoding**. Das One-Hot Encoding erzeugt für jede mögliche Kategorie jedes nicht numerischen Features eine _„Dummy“_-Variable.

Außerdem müssen wir, wie bei den nicht numerischen Features, die nicht numerische Zielvariable `'income'` in numerische Werte umwandeln, damit der Lernalgorithmus funktioniert. Da es für dieses Label nur zwei mögliche Kategorien gibt ("<=50K" und ">50K"), können wir auf One-Hot Encoding verzichten und diese beiden Kategorien einfach als `0` bzw. `1` kodieren.

In [ ]:
# Daten in 'features' mit pandas.get_dummies() per One-Hot Encoding kodieren

# Daten in 'income'/'target' als numerische Werte kodieren

# Anzahl der Features nach dem One-Hot Encoding ausgeben

### Daten mischen und aufteilen

Jetzt sind alle _kategorialen Variablen_ in numerische Features umgewandelt. Als Nächstes teilen wir die Daten (sowohl die Features als auch ihre Labels) in einen Trainings- und einen Testdatensatz auf. 80% der Daten verwenden wir für das Training und 20% für den Test.

In [ ]:
from sklearn.model_selection import train_test_split

# Daten aus 'features' und 'income' in Trainings- und Testdatensatz aufteilen

# Ergebnisse der Aufteilung anzeigen
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))

### Numerische Features normalisieren
Es empfiehlt sich oft, numerische Features in irgendeiner Form zu skalieren. Eine Skalierung ändert die Form der Verteilung eines Features nicht (etwa bei `'capital-gain'` oder `'capital-loss'` oben); die Normalisierung sorgt aber dafür, dass jedes Feature gleich behandelt wird, wenn Algorithmen für überwachtes Lernen angewendet werden. Beachte, dass die Daten nach der Skalierung in ihrer Rohform nicht mehr dieselbe ursprüngliche Bedeutung haben, wie das Beispiel unten zeigt. Um Datenlecks zu vermeiden, normalisieren wir die Daten erst, nachdem wir sie in Trainings- und Testdatensatz aufgeteilt haben.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Scaler initialisieren und dann auf die Features anwenden

# Beispiel eines Eintrags nach der Skalierung anzeigen

----
## Modellleistung evaluieren
In diesem Abschnitt untersuchen wir vier verschiedene Algorithmen und bestimmen, welcher die Daten am besten modelliert.

### Metriken und das naive Vorhersagemodell

*CharityML* weiß aus eigener Recherche, dass Personen, die mehr als 50.000 \$ verdienen, am ehesten an die Organisation spenden. Deshalb ist *CharityML* besonders daran interessiert, treffsicher vorherzusagen, wer mehr als 50.000 \$ verdient. Es scheint angemessen, die **Accuracy** als Metrik für die Evaluation der Leistung eines bestimmten Modells zu verwenden. Außerdem wäre es für *CharityML* schädlich, jemanden, der *nicht* mehr als 50.000 \$ verdient, als jemanden einzustufen, der es tut, denn die Organisation sucht Personen, die bereit sind zu spenden. Deshalb ist die Fähigkeit eines Modells, präzise vorherzusagen, wer mehr als 50.000 \$ verdient, *wichtiger* als sein **Recall** für diese Personen. Als Metrik, die sowohl Precision als auch Recall berücksichtigt, können wir den **F-beta-Score** verwenden:

$$ F_{\beta} = (1 + \beta^2) \cdot \frac{precision \cdot recall}{\left( \beta^2 \cdot precision \right) + recall} $$

Insbesondere wird bei $\beta = 0.5$ die Precision stärker gewichtet.

Ein Blick auf die Klassenverteilung (Personen, die höchstens 50.000 \$ verdienen, und Personen, die mehr verdienen) zeigt deutlich, dass die meisten Personen nicht mehr als 50.000 \$ verdienen. Das kann die **Accuracy** stark beeinflussen, denn wir könnten einfach sagen „*Diese Person verdient nicht mehr als 50.000 \$*“ und hätten meistens recht, ohne uns die Daten je angesehen zu haben! Eine solche Aussage gilt als **naiv**, da wir keinerlei Informationen berücksichtigt haben, die die Behauptung stützen. Es ist immer wichtig, die *naive Vorhersage* für deine Daten zu betrachten, um einen Benchmark dafür festzulegen, ob ein Modell gut abschneidet. Trotzdem wäre es sinnlos, diese Vorhersage zu verwenden: Wenn wir für alle Menschen vorhersagen würden, dass sie weniger als 50.000 \$ verdienen, würde *CharityML* keine Spendenden finden.

### Leistung des naiven Vorhersagemodells

Angenommen, wir wählen ein Modell, das immer vorhersagt, dass eine Person mehr als 50.000 \$ verdient: Welche Accuracy und welchen F-Score hätte dieses Modell auf diesem Datensatz? (Denk daran, was du am Anfang des Notebooks berechnet hast.)

In [ ]:
# TODO: Accuracy berechnen
accuracy = 

# TODO: Precision berechnen
precision =

# TODO: Recall berechnen
recall = 

# TODO: F-Score mit der Formel oben für beta = 0.5 berechnen
fscore =  

# Ergebnisse ausgeben 
print ("Naive Predictor: [Accuracy score: {:.4f}, F-score: {:.4f}]".format(accuracy, fscore))

##  Modelle für überwachtes Lernen

###  Anwendung der Modelle

Jetzt wählen wir zwei der oben genannten Modelle für überwachtes Lernen aus, die für dieses Problem geeignet sind, und testen sie auf den Zensusdaten.

**Entscheidungsbäume**

 - Anwendung in der Praxis: Entscheidungsbäume und allgemein CART (Classification and Regression Trees) werden oft in der Finanzanalyse eingesetzt. Ein konkretes Beispiel ist, anhand der bisherigen Wertentwicklung vorherzusagen, welche Aktien gekauft werden sollten. [Referenz](https://ir.nctu.edu.tw/bitstream/11536/11962/1/000237645100007.pdf)
 - Stärken:
      - Können kategoriale und numerische Daten verarbeiten.
      - Brauchen wenig Datenvorverarbeitung und kommen mit Daten zurecht, die nicht normalisiert oder nicht passend für das Machine Learning kodiert wurden.
      - Einfach zu verstehen und zu interpretieren.
 - Schwächen:
     - Komplexe Entscheidungsbäume generalisieren schlecht auf die Daten und können zu Overfitting führen.
     - Instabil, da kleine Abweichungen in den Daten zu einem anderen Entscheidungsbaum führen können. Deshalb werden sie meist in einem Ensemble (wie Random Forests) eingesetzt, um Robustheit zu erreichen.
     - Können verzerrte Bäume erzeugen, wenn einige Klassen dominieren.
 - Eignung: Da ein Entscheidungsbaum sowohl numerische als auch kategoriale Daten verarbeiten kann, ist er ein guter Kandidat für unseren Fall (auch wenn die Vorverarbeitungsschritte einen möglichen Vorteil bereits aufheben könnten). Er ist außerdem leicht zu interpretieren, sodass wir wissen, was unter der Haube passiert, wenn wir die Ergebnisse interpretieren.


**Ensemble-Methoden: AdaBoost**

 - Anwendung in der Praxis: Ensemble-Methoden werden in Kaggle-Wettbewerben häufig eingesetzt, besonders in der Bilderkennung. Ein Beispiel aus der Praxis für AdaBoost ist die Objekterkennung in Bildern, etwa das Erkennen der Spielenden während eines Basketballspiels. [Referenz](https://uni-obuda.hu/journal/Markoski_Ivankovic_Ratgeber_Pecev_Glusac_57.pdf)
 - Stärken:
     - Ensemble-Methoden, darunter AdaBoost, sind robuster als einzelne Estimators und haben eine verbesserte Verallgemeinerbarkeit.
     - Einfache Modelle lassen sich zu einem komplexen Modell kombinieren, das rechnerisch trotzdem schnell bleibt.
 - Schwächen:
     - Wenn wir einen verzerrten zugrunde liegenden Klassifikator haben, führt das zu einem verzerrten Boosting-Modell.
 - Eignung: Ensemble-Methoden gelten als hochwertige Klassifikatoren, und AdaBoost ist einer der beliebtesten Boosting-Algorithmen. Außerdem haben wir in unserem Datensatz ein Klassenungleichgewicht, mit dem Boosting robust umgehen könnte.


In [ ]:
# Benötigte Metriken und Modelle importieren
from sklearn.metrics import confusion_matrix
from sklearn.metrics import fbeta_score, accuracy_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

### Modelle trainieren und vergleichen

Im nächsten Schritt trainieren, evaluieren und vergleichen wir verschiedene Modelle. Du kannst entscheiden, ob du den Code mehrmals schreiben oder es eleganter lösen willst, indem du eine Funktion definierst, die die Arbeit für dich erledigt.

In [ ]:
# Einen Entscheidungsbaum und einen AdaBoostClassifier auf den Trainingsdaten trainieren

### Modellevaluation

Jetzt können wir die Leistung unserer Modelle anhand der Accuracy und des F1-Scores evaluieren und die Confusion Matrix für die Ergebnisse visualisieren.

In [ ]:
# Die verschiedenen Modelle evaluieren und dabei Accuracy und F1-Score vergleichen

In [ ]:
# Confusion Matrix für jeden Klassifikator visualisieren

### Modell-Tuning
Mit Grid Search (`GridSearchCV`) und verschiedenen Kombinationen aus Parametern und Werten können wir unser Modell für noch bessere Ergebnisse tunen. Wir tunen den AdaBoostClassifier, da er die beste Leistung gezeigt hat.
Bei AdaBoost tunen wir die Parameter n_estimators und learning rate sowie die Parameter des Basisklassifikators (denk daran, dass unser Basisklassifikator für das AdaBoost-Ensemble ein Entscheidungsbaum ist!).

In [ ]:
# 'GridSearchCV', 'make_scorer' und alle weiteren benötigten Bibliotheken importieren
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

# Klassifikator initialisieren
clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42,
)

# Liste der Parameter erstellen, die du tunen möchtest
parameters = {
    "n_estimators": [50, 120],
    "learning_rate": [0.1, 0.5, 1.0],
    "estimator__min_samples_split": np.arange(2, 8, 2),
    "estimator__max_depth": np.arange(1, 4, 1),
}

# Scoring-Objekt mit fbeta_score erstellen
scorer = make_scorer(fbeta_score, beta=0.5)

# Grid Search auf dem Klassifikator mit 'scorer' als Scoring-Methode durchführen

# Grid-Search-Objekt auf den Trainingsdaten trainieren und die optimalen Parameter finden

# Den besten Estimator abrufen
best_clf =
# Vorhersagen mit dem nicht optimierten Modell erstellen
predictions = 
# Vorhersagen mit dem optimierten Modell erstellen
best_predictions = 

# Scores vorher und nachher ausgeben
print("Un-optimized model\n------")
print(
    "Accuracy score on testing data: {:.4f}".format(accuracy_score(y_test, predictions))
)
print(
    "F-score on testing data: {:.4f}".format(fbeta_score(y_test, predictions, beta=0.5))
)
print("\nOptimized Model\n------")
print(
    "Final accuracy score on the testing data: {:.4f}".format(
        accuracy_score(y_test, best_predictions)
    )
)
print(
    "Final F-score on the testing data: {:.4f}".format(
        fbeta_score(y_test, best_predictions, beta=0.5)
    )
)
print(best_clf)